<a href="https://colab.research.google.com/github/dan-zeman/belo-horizonte/blob/main/Udapi2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Universal Dependencies with `udapi-python`

This is the second session with the `udapi-python` library. As we start with a blank virtual machine, we must start again with installing Udapi and downloading the treebank data.

## 1. Install `udapi`

First, we need to install the `udapi` Python library. This can be done using `pip`.

In [1]:
%%bash
# (Preceding a single line with ! makes that line interpreted by shell instead of python.
# Inserting %%bash at the beginning of the cell makes the whole cell interpreted by bash.)
pip install --upgrade udapi
udapy -h

usage: udapy [optional_arguments] scenario

udapy - Python interface to Udapi - API for Universal Dependencies

Examples of usage:
  udapy -s read.Sentences udpipe.En < in.txt > out.conllu
  udapy -T < sample.conllu | less -R
  udapy -HAM ud.MarkBugs < sample.conllu > bugs.html

positional arguments:
  scenario              A sequence of blocks and their parameters.

options:
  -h, --help            show this help message and exit
  -q, --quiet           Warning, info and debug messages are suppressed. Only fatal errors are reported.
  -v, --verbose         Warning, info and debug messages are printed to the STDERR.
  -s, --save            Add write.Conllu to the end of the scenario
  -T, --save_text_mode_trees
                        Add write.TextModeTrees color=1 to the end of the scenario
  -H, --save_html       Add write.TextModeTreesHtml color=1 to the end of the scenario
  -A, --save_all_attributes
                        Add attributes=form,lemma,upos,xpos,feats,deprel,misc (to

## 2. Download and Load a UD Treebank

Download one or more UD treebanks from GitHub. Feel free to add downloads of other treebanks you are interested in. See https://universaldependencies.org/ for the list of available treebanks. Besides the UD homepage, you can also try https://universaldependencies.org/languages.html, where you can filter languages by family and genus.

Smart tip: If you want to work with your own data instead, click on the Files icon in the left pane of Colab, create a treebank folder on the local disk and upload your CoNLL-U file(s) there. Then you will be able to say that this is the treebank you want to load and process.

In [2]:
%%bash
rm -rf UD_*
git clone https://github.com/UniversalDependencies/UD_Portuguese-Porttinari.git

Cloning into 'UD_Portuguese-Porttinari'...


Now we will access the treebank data from Python. The `udapi.core.document.Document` class is used to load the treebank. You can edit the code cell below and specify the treebank you want to work with (it must be one of the treebanks you downloaded above).

In [3]:
import glob
from udapi.core.document import Document

# Read the CoNLL-U file.
###############################################################################
# TODO: Change this variable to explore different datasets.
treebank = 'UD_Portuguese-Porttinari' # Student can modify this line.

# List all .conllu files in the specified treebank folder.
conllu_files = glob.glob(f"{treebank}/*.conllu")
print(f"Found {len(conllu_files)} CoNLL-U files in {treebank}:")
for file in sorted(conllu_files):
    print(file)
print(f"Loading {treebank}...")
# Each CoNLL-U file will be stored as one Document object.
# Note that some treebanks use the '# newdoc' comment to mark document boundaries within each file.
# That is a different notion of document!
filedocs = []
nsent = 0
for file in sorted(conllu_files):
    doc = Document(filename=file)
    filedocs.append(doc)
    nsent += len(doc.bundles)
print(f"Loaded {nsent} sentences from {treebank}.")


Found 3 CoNLL-U files in UD_Portuguese-Porttinari:
UD_Portuguese-Porttinari/pt_porttinari-ud-dev.conllu
UD_Portuguese-Porttinari/pt_porttinari-ud-test.conllu
UD_Portuguese-Porttinari/pt_porttinari-ud-train.conllu
Loading UD_Portuguese-Porttinari...
Loaded 8418 sentences from UD_Portuguese-Porttinari.


## 3. Get the Trees

In the previous session, we were collecting counts of various observations in the treebank, and displaying them in a table. But can we use Udapi also to display the trees with examples? Yes, we can!

Simple searches can be achieved by calling the so-called command-line interface of Udapi; it means that we are calling Udapi as a standalone program (which is actually spelled `udapy`, note the final letter).

In [46]:
from google.colab import files

# Run udapy as a standalone program.
# The first part of the command sends our CoNLL-U files as input to the pipeline.
# The util.Mark node='...' part will mark each node that satisfies the condition.
# The -HAM options mean that udapy will output the trees as colored HTML,
# highlighting the marked nodes. The output will contain only trees that contain
# at least one marked node.
!cat UD_Portuguese-Porttinari/*.conllu | udapy -HAM util.Mark node='node.udeprel == "expl" and node.parent.upos == "VERB"' > udapi_output.html

# Offer the file for download so that you can open it in your browser.
# We could display it here in Colab but it would not be practical because the
# file could be very long.
print(f"File 'udapi_output.html' is ready for download.")
files.download('udapi_output.html')


2026-07-23 04:02:52,579 [   INFO] run_blocks - No reader specified, using read.Conllu
2026-07-23 04:02:52,579 [   INFO] run_blocks -  ---- ROUND ----
2026-07-23 04:02:52,580 [   INFO] run_blocks - Executing block read.Conllu {}
2026-07-23 04:02:52,580 [   INFO] try_fast_load - Reading -
2026-07-23 04:02:53,440 [   INFO] run_blocks - Executing block util.Mark node=node.udeprel == "expl" and node.parent.upos == "VERB"
2026-07-23 04:02:56,360 [   INFO] run_blocks - Executing block write.TextModeTreesHtml color=1 attributes=form,lemma,upos,xpos,feats,deprel,misc marked_only=1
File 'udapi_output.html' is ready for download.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Text Mode, One Tree at a Time

We can also ask Udapi to draw the tree structure if we are using Udapi's functions from within Python. In that case, we use the function `root.draw()`.

Here we have two code cells. The first one defines a function that will find the next sentence containing at least one node with a given UPOS (`search_upos`), mark those nodes by adding a temporary attribute Mark to their annotation, and return the sentence so that it can be displayed. Further down, the cell says that our `search_upos` is `VERB`. We need to run this cell only once, as we do with most cells in the notebook.

The subsequent cell is different. We can run it many times and each time it will deliver the next example sentence, until we reach the end of the treebank (or until we run the initialization cell again).

In [54]:
def find_and_yield_sentences(filedocs, search_upos):
    """
    A generator function that yields sentences (bundles) containing a node
    with the specified Universal Part-of-Speech (UPOS) tag.
    """
    for doc in filedocs:
        for bundle in doc.bundles:
            hits = 0
            # Get all nodes in the current sentence (bundle)
            nodes = bundle.get_tree().descendants
            for node in nodes:
                # Check if any node in the sentence matches the search_upos
                if node.upos == search_upos:
                    node.misc['Mark'] = '<==============='
                    hits += 1
            if hits > 0:
                yield bundle # Yield the entire bundle (sentence) when found

# Example usage of the generator:
# Let's search for sentences containing a verb (VERB UPOS).
# Initialize the generator. This creates the iterator object.
# This object will maintain its state across calls to `next()`.
################################################################################
# TODO: You can replace 'VERB' with another UPOS (or modify the condition inside the generator).
verb_sentence_iterator = find_and_yield_sentences(filedocs, 'VERB')

print("Initialized generator to find sentences with 'VERB'.")

Initialized generator to find sentences with 'VERB'.


In [70]:
try:
    # Get the next sentence from the generator
    next_verb_sentence = next(verb_sentence_iterator)
    root = next_verb_sentence.get_tree()

    # Display the tree structure using root.draw()
    root.draw(attributes='form,lemma,upos,feats,deprel,misc[Mark]')

except StopIteration:
    print("No more results.")
    print("Run the initialization cell again (the one defining `verb_sentence_iterator`) to start from the beginning.")

# sent_id = FOLHA_DOC002013_SENT006
# text = O aplicativo é comparado a um "GPS" que ajuda a navegar a estrutura das composições eruditas.
─┮
 │   ╭─╼ O o DET Definite=Def|Gender=Masc|Number=Sing|PronType=Art det 
 │ ╭─┶ aplicativo aplicativo NOUN Gender=Masc|Number=Sing nsubj:pass 
 │ ┢─╼ é ser AUX Mood=Ind|Number=Sing|Person=3|Tense=Pres|VerbForm=Fin aux:pass 
 ╰─┾ comparado comparar VERB Gender=Masc|Number=Sing|VerbForm=Part|Voice=Pass root <===============
   │ ╭─╼ a a ADP _ case 
   │ ┢─╼ um um DET Definite=Ind|Gender=Masc|Number=Sing|PronType=Art det 
   │ ┢─╼ " " PUNCT _ punct 
   ┡─┾ GPS GPS PROPN _ obl 
   │ ┡─╼ " " PUNCT _ punct 
   │ │ ╭─╼ que que PRON PronType=Rel nsubj 
   │ ╰─┾ ajuda ajudar VERB Mood=Ind|Number=Sing|Person=3|Tense=Pres|VerbForm=Fin acl:relcl <===============
   │   │ ╭─╼ a a ADP _ mark 
   │   ╰─┾ navegar navegar VERB VerbForm=Inf xcomp <===============
   │     │ ╭─╼ a o DET Definite=Def|Gender=Fem|Number=Sing|PronType=Art det 
   │     ╰─┾ estrutura est

## 4. Some Technical Code to Enable Clustering Tables

You don't need to understand or modify the following cell but you must run it so that we can display nice clustering tables later.

In [6]:
from collections import defaultdict
import pandas as pd

def generate_two_dimensional_table(clusters, display_mode='#'):
    """
    Generates, sorts, and displays a two-dimensional table from a clusters dictionary.

    Args:
        clusters (dict): A dictionary where keys are (parent_tag, child_tag) tuples
                         and values are counts.
        display_mode (str): The mode for displaying cell values.
                            Options: '#': absolute counts,
                                     'H%': horizontal percentage (row-wise),
                                     'V%': vertical percentage (column-wise),
                                     '%': percentage of grand total.
    Returns:
        pandas.DataFrame: The DataFrame used for display.
    """
    import pandas as pd

    # Get all unique parent and child tags
    parent_tags = sorted(list(set([pair[0] for pair in clusters.keys()])))
    child_tags = sorted(list(set([pair[1] for pair in clusters.keys()])))

    # Create a DataFrame to store the counts (initial data only)
    df_raw_counts = pd.DataFrame(0, index=parent_tags, columns=child_tags)

    # Populate the DataFrame with counts from the clusters dictionary
    for (parent, child), count in clusters.items():
        df_raw_counts.loc[parent, child] = count

    # Calculate row sums for sorting purposes
    row_sums_for_sorting = df_raw_counts.sum(axis=1)

    # Sort rows based on these sums in descending order
    sorted_row_indices = row_sums_for_sorting.sort_values(ascending=False).index
    df_raw_counts = df_raw_counts.reindex(sorted_row_indices)

    # Calculate column sums for sorting purposes (after row sorting)
    column_sums_for_sorting = df_raw_counts.sum(axis=0)

    # Sort columns based on these sums in descending order
    sorted_column_names = column_sums_for_sorting.sort_values(ascending=False).index
    df_raw_counts = df_raw_counts.reindex(columns=sorted_column_names)

    # --- Prepare DataFrame for display based on display_mode ---
    df_display = df_raw_counts.copy() # Start with raw counts

    # Calculate absolute totals for later use (these will always be absolute counts)
    absolute_row_totals = df_raw_counts.sum(axis=1)
    absolute_column_totals = df_raw_counts.sum(axis=0)
    absolute_grand_total = df_raw_counts.sum().sum()

    if display_mode == 'H%':
        # Horizontal percentage: percent of the total count of the row
        df_display = df_raw_counts.div(absolute_row_totals.replace(0, 1), axis=0) * 100
        df_display = df_display.round(2).fillna(0)
        df_display = df_display.map(lambda x: f'{x:.2f}%')
    elif display_mode == 'V%':
        # Vertical percentage: percent of the total count of the column
        df_display = df_raw_counts.div(absolute_column_totals.replace(0, 1), axis=1) * 100
        df_display = df_display.round(2).fillna(0)
        df_display = df_display.map(lambda x: f'{x:.2f}%')
    elif display_mode == '%':
        # Percent of the total count in the table
        df_display = (df_raw_counts / absolute_grand_total) * 100
        df_display = df_display.round(2).fillna(0)
        df_display = df_display.map(lambda x: f'{x:.2f}%')
    elif display_mode == '#':
        # Absolute counts (already in df_raw_counts, convert to string for display consistency)
        df_display = df_raw_counts.astype(str)
    else:
        raise ValueError("Invalid display_mode. Choose from '#', 'H%', 'V%', '%'.")

    # Add the absolute Row_Total column
    df_display['Total'] = absolute_row_totals.astype(str)

    # Add the absolute Col_Total row
    col_total_row_for_display = absolute_column_totals.astype(str)
    col_total_row_for_display['Total'] = str(absolute_grand_total) # Grand total
    df_display.loc['Total'] = col_total_row_for_display

    return df_display